In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt

In [2]:
# Функция для вычисления log-loss на каждой итерации бустинга
def compute_losses(model, X_train, y_train, X_test, y_test):
    losses_train = []
    losses_test = []
    # staged_decision_function возвращает сырые оценки
    for pred_raw in model.staged_decision_function(X_train):
        prob = 1 / (1 + np.exp(-pred_raw))
        loss = log_loss(y_train, prob)
        losses_train.append(loss)
    for pred_raw in model.staged_decision_function(X_test):
        prob = 1 / (1 + np.exp(-pred_raw))
        loss = log_loss(y_test, prob)
        losses_test.append(loss)
    return np.array(losses_train), np.array(losses_test)

In [3]:
# 1. Загрузка данных
data = pd.read_csv('gbm-data.csv').values
X, y = data[:, 1:], data[:, 0]

# Разбиение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=241)

results = {}

# 2. Перебор learning_rate и обучение
for lr in [1, 0.5, 0.3, 0.2, 0.1]:
    gbm = GradientBoostingClassifier(
        n_estimators=250,
        learning_rate=lr,
        verbose=False,
        random_state=241
    )
    gbm.fit(X_train, y_train)
    train_loss, test_loss = compute_losses(gbm, X_train, y_train, X_test, y_test)
    min_idx = np.argmin(test_loss)
    min_loss = test_loss[min_idx]
    results[lr] = {
        'train_loss': train_loss,
        'test_loss': test_loss,
        'min_loss': min_loss,
        'min_iter': min_idx + 1  # нумерация с 1
    }
    print(f"lr={lr}: min test loss = {min_loss:.4f}, iter {min_idx+1}")

lr=1: min test loss = 0.5823, iter 1
lr=0.5: min test loss = 0.5581, iter 7
lr=0.3: min test loss = 0.5433, iter 11
lr=0.2: min test loss = 0.5299, iter 37
lr=0.1: min test loss = 0.5258, iter 52


In [4]:
lr = 0.2
train_loss = results[lr]['train_loss']
test_loss = results[lr]['test_loss']

# Графики
plt.figure()
plt.plot(train_loss, 'g', linewidth=2, label='train')
plt.plot(test_loss, 'r', linewidth=2, label='test')
plt.legend()
plt.xlabel('Iteration')
plt.ylabel('Log-loss')
plt.title(f'GBM learning_rate={lr}')
plt.savefig('gbm_loss_curve.png')
plt.close()

# 4. Минимальное значение log-loss и номер итерации
min_loss = results[lr]['min_loss']
min_iter = results[lr]['min_iter']

# 5. Случайный лес с числом деревьев = min_iter
rf = RandomForestClassifier(n_estimators=min_iter, random_state=241)
rf.fit(X_train, y_train)
y_pred_proba = rf.predict_proba(X_test)[:, 1]  # вероятность класса 1
rf_loss = log_loss(y_test, y_pred_proba)


print(f"Random Forest log-loss with {min_iter} trees: {rf_loss:.4f}")

Random Forest log-loss with 37 trees: 0.5411


In [5]:
with open('answer1.txt', 'w') as f:
    f.write('overfitting')

with open('answer2.txt', 'w') as f:
    f.write(f"{min_loss:.2f} {min_iter}")

with open('answer3.txt', 'w') as f:
    f.write(f"{rf_loss:.2f}")